<a href="https://colab.research.google.com/github/martinruhle/curso-ciencia-datos-2027-1/blob/lab02/lab02/analisis_pacientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 02 — Análisis de un dataset de pacientes con presupuesto de memoria

**Actividades 1 a 9.** Dataset sintético generado con Synthea (**23 018 pacientes**: 20 001 vivos + 3 017 fallecidos; 17 355 578 observaciones).

El laboratorio evalúa criterio, no que el código corra: cada dtype, cada conversión y cada elección de herramienta están justificados en el texto.

## 0. Entorno y datos

### 0.1 Librerías
`polars`, `pyspark`, `memory_profiler` y `pyarrow` no vienen preinstalados en Colab. `gc` se usa para liberar RAM entre la carga ingenua y la tipada de `observations`, que es la tabla pesada (17.4 M filas).

In [1]:
!pip install -q polars pyspark memory_profiler pyarrow psutil

### 0.2 Arranque, rutas y checkpoint
Colab borra el disco al reiniciar la sesión y el runtime se cae por inactividad, así que reconstruir el estado desde cero cuesta varios minutos. La celda siguiente resuelve tres cosas de una vez:

1. **Importa y monta Drive.** Es autosuficiente a propósito: no depende de ninguna celda anterior, así que puede ejecutarse sola después de una desconexión. `drive.mount` es idempotente — si ya está montado, no hace nada.
2. **Fija las rutas.** `DATA` guarda los CSV de Synthea; `CKPT` guarda los checkpoints. Ambas viven en Drive, no en el disco efímero de Colab.
3. **Restaura el estado si existe checkpoint.** Los DataFrames ya tipados se persisten en **Parquet**, que guarda el esquema junto con los datos: al recargar, `category`, `datetime64[ns, UTC]` y `float32` vuelven intactos. Un CSV no puede hacer eso — obligaría a rehacer toda la Actividad 2 en cada reconexión.

`FORCE_REBUILD` controla el comportamiento:
- `False` — si hay checkpoint, lo carga y permite saltar a la actividad en curso. Es el modo de trabajo diario.
- `True` — ignora el caché y ejecuta el pipeline completo desde los CSV.

> **Para la entrega se corre con `FORCE_REBUILD = True` y *Restart & Run All*.** El caché acelera el trabajo pero puede enmascarar un pipeline roto, y el criterio de evaluación exige que el notebook corra completo de cero. Además, el checkpoint solo contiene `obs`, `p8` y `enc_tip`: los objetos intermedios (`obs_naive`, `chk`, `m1`, `m2`) se reconstruyen ejecutando las celdas en orden.

In [2]:
import pandas as pd, numpy as np, gc
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/curso-cd-2027/lab02')
CKPT = BASE / 'ckpt'
DATA = BASE / 'data'
DATA.mkdir(parents=True, exist_ok=True)

FORCE_REBUILD = True   # True = ignorar caché y correr el pipeline completo
usar_cache = (not FORCE_REBUILD) and (CKPT/'obs.parquet').exists()

if usar_cache:
    obs     = pd.read_parquet(CKPT/'obs.parquet')
    p8      = pd.read_parquet(CKPT/'p8.parquet')
    enc_tip = pd.read_parquet(CKPT/'enc_tip.parquet')
    print('estado restaurado desde checkpoint')
    print(obs.dtypes)        # verificá que category/float32 sobrevivieron
else:
    print('sin caché: corré las celdas de Actividades 0-3')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sin caché: corré las celdas de Actividades 0-3


### 0.3 Generación con Synthea

Se descarga el jar `with-dependencies` desde el tag `master-branch-latest` (siempre apunta a la última versión). Dos decisiones en el comando de generación:

- `--exporter.csv.export true` — enciende la exportación a CSV, apagada por defecto.
- `--exporter.fhir.export false` — apaga FHIR, que no se usa en este lab y triplica el tiempo.

`-p 20001` fija la población **viva**; Synthea además simula a quienes murieron en la ventana temporal, así que `patients.csv` termina con **23 018 filas** (20 001 vivas + 3 017 fallecidas). El comando de abajo es el exacto usado, anotado por reproducibilidad.

**Ejecución condicional.** La generación corre únicamente si faltan los CSV en Drive. Regenerar en cada ejecución sería contraproducente: Synthea no es determinista sin semilla fija, de modo que una nueva corrida produciría una cohorte distinta y los números de la narrativa dejarían de corresponder a los datos analizados.

**Limitación de reproducibilidad declarada.** El comando no fija la semilla aleatoria, por lo que reproduce el procedimiento pero no la cohorte exacta: quien lo ejecute obtendrá 23 018 pacientes de características estadísticas equivalentes, no los mismos individuos. Fijar la semilla (parámetro -s de Synthea) haría la generación bit a bit reproducible y es lo recomendable para un trabajo que deba replicarse exactamente.

In [5]:
# Los CSV se generan UNA sola vez y persisten en Drive. Esta celda solo actúa si faltan,
# de modo que un Restart & Run All no regenere la cohorte ni altere los datos analizados.
NECESARIOS = ['patients.csv', 'encounters.csv', 'observations.csv']
faltan = [f for f in NECESARIOS if not (DATA/f).exists()]

if faltan:
    print('Faltan CSV en Drive, generando con Synthea:', faltan)
    !java -version 2>&1 | head -1 || apt-get -qq install -y openjdk-17-jdk-headless
    !wget -q https://github.com/synthetichealth/synthea/releases/download/master-branch-latest/synthea-with-dependencies.jar -O synthea-with-dependencies.jar
    !java -jar synthea-with-dependencies.jar -p 20001 --exporter.csv.export true --exporter.fhir.export false
    !cp output/csv/*.csv "{DATA}/"
else:
    print('CSV ya presentes en Drive; se omite la generación.')

CSV ya presentes en Drive; se omite la generación.


In [6]:
# Verificación: falla ruidosamente si falta algo, en vez de romper tres celdas más abajo.
faltantes = [f for f in NECESARIOS if not (DATA/f).exists()]
assert not faltantes, f'Faltan archivos en {DATA}: {faltantes}'

for f in NECESARIOS:
    print(f'{f:20s} {(DATA/f).stat().st_size/1e6:9.1f} MB')

patients.csv               6.9 MB
encounters.csv           452.8 MB
observations.csv        3058.5 MB


## Actividad 1 — Carga con dtypes explícitos

Cargamos cada archivo dos veces —primero con la inferencia de pandas, después declarando los dtypes— y medimos el consumo en cada caso.

### Clasificación de columnas (`patients.csv`)
El dtype sale de clasificar cada columna en una de tres cubetas:

| Columna | Cubeta | dtype |
|---|---|---|
| `Id` | identificador (único, 1 por fila) | `object` |
| `BIRTHDATE` | fecha | `datetime64` |
| `DEATHDATE` | fecha | `datetime64` |
| `GENDER` | categórica | `category` |
| `RACE` | categórica | `category` |
| `ETHNICITY` | categórica | `category` |
| `CITY` | categórica (mayor cardinalidad) | `category` |
| `STATE` | categórica (cardinalidad 1: todo Massachusetts) | `category` |

> Sobre `CODE` (en encounters/observations): *parece* identificador, pero es un código clínico (SNOMED/LOINC) con pocas decenas de valores que se repiten en millones de filas → `category`, no `object`.

### Medición de memoria
`memory_usage(deep=True)` es la medición correcta: sin `deep`, una columna `object` reporta solo el peso de los punteros (8 bytes por celda), no los bytes reales de cada string. En columnas `object` la diferencia es de órdenes de magnitud.

In [7]:
def mb(df):
    """Memoria real del DataFrame en MB."""
    return df.memory_usage(deep=True).sum() / 1e6

### Carga ingenua (baseline)
Primera pasada: dtypes inferidos, anotando el consumo de los tres archivos. El `del ... gc.collect()` libera antes de seguir — clave con `observations`.

In [8]:
naive = {}
for name in ['patients', 'encounters', 'observations']:
    df = pd.read_csv(DATA / f'{name}.csv')     # inferencia por defecto
    naive[name] = mb(df)
    print(f'{name:14s} {naive[name]:8.1f} MB   {df.shape}')
    del df; gc.collect()

patients           28.3 MB   (23018, 28)
encounters       1099.1 MB   (1355775, 15)
observations    10454.3 MB   (17355578, 9)


### Tabla comparativa: crudo (28 col) vs filtrado (8 col) vs tipado
`usecols` filtra **en la lectura**, no después. Comparamos las 28 columnas que exporta Synthea contra las 8 que pide el lab, y esas 8 contra su versión tipada. `enc_naive` queda cargado para reutilizarlo en la Actividad 3.

In [9]:
mem = []  # filas de la tabla comparativa

# ---- patients ----
cols_patients = ['Id','BIRTHDATE','DEATHDATE','GENDER','RACE','ETHNICITY','CITY','STATE']
dtypes_patients = {'Id':'object','GENDER':'category','RACE':'category',
                   'ETHNICITY':'category','CITY':'category','STATE':'category'}
# BIRTHDATE/DEATHDATE van en parse_dates, NO en dtype (no pueden estar en ambos)

p_raw   = pd.read_csv(DATA/'patients.csv')                       # 28 col, inferido
mem.append(('patients','crudo 28col', mb(p_raw), p_raw.shape));  del p_raw; gc.collect()

p8_naive = pd.read_csv(DATA/'patients.csv', usecols=cols_patients)   # 8 col, inferido
mem.append(('patients','8col inferido', mb(p8_naive), p8_naive.shape))

p8 = pd.read_csv(DATA/'patients.csv', usecols=cols_patients,
                 dtype=dtypes_patients, parse_dates=['BIRTHDATE','DEATHDATE'])  # 8 col tipado
mem.append(('patients','8col tipado', mb(p8), p8.shape))

# ---- encounters ----  mismo criterio: ID=object, categóricas, fechas
cols_enc = ['Id','START','STOP','PATIENT','ENCOUNTERCLASS','CODE','DESCRIPTION']
enc_naive = pd.read_csv(DATA/'encounters.csv', usecols=cols_enc)
mem.append(('encounters','inferido', mb(enc_naive), enc_naive.shape))

dtypes_enc = {'Id':'object','PATIENT':'category','ENCOUNTERCLASS':'category',
              'CODE':'category','DESCRIPTION':'category'}   # PATIENT repite: category
enc_tip = pd.read_csv(DATA/'encounters.csv', usecols=cols_enc,
                      dtype=dtypes_enc, parse_dates=['START','STOP'])
mem.append(('encounters','tipado', mb(enc_tip), enc_tip.shape))

# ---- observations ----  solo baseline; la reducción es la Act.2
obs_naive = pd.read_csv(DATA/'observations.csv')
mem.append(('observations','inferido', mb(obs_naive), obs_naive.shape))

tabla1 = pd.DataFrame(mem, columns=['archivo','version','MB','shape'])
tabla1

,archivo,version,MB,shape
0,patients,crudo 28col,28.292863,"(23018, 28)"
1,patients,8col inferido,10.645683,"(23018, 8)"
2,patients,8col tipado,2.505041,"(23018, 8)"
3,encounters,inferido,622.489679,"(1355775, 7)"
4,encounters,tipado,146.211358,"(1355775, 7)"
5,observations,inferido,10454.268422,"(17355578, 9)"


**Lectura de la tabla.** Conviene comparar en **MB absolutos**, no en porcentajes: cada porcentaje está calculado sobre una base distinta y compararlos entre sí lleva a una conclusión equivocada.

Para `patients`: 28.29 MB en crudo → 10.65 MB al filtrar a las 8 columnas → 2.51 MB al declarar los dtypes. El filtrado ahorra **17.65 MB** y el tipado **8.14 MB**. Es decir, **`usecols` aporta más del doble de ahorro absoluto que la declaración de dtypes**, aunque su porcentaje suene menor (62.4% del crudo, frente al 76.5% que el tipado recorta sobre una base ya reducida).

La conclusión práctica: el primer ahorro y el más barato es **no leer lo que no se necesita**. Los dtypes actúan después, sobre lo que quedó. Para `encounters` el tipado baja de 622.49 MB a 146.21 MB (76.5%), sin línea de comparación en crudo porque se leyó directamente con `usecols`.

`observations` se analiza en la Actividad 2, donde la reducción es el objetivo explícito.

## Actividad 2 — Reducir la memoria de `observations` ≥ 70%

### Diagnóstico antes de decidir
Antes de convertir nada: cardinalidad por columna (decide dónde `category` ayuda) y qué fracción de `VALUE` no es numérica.

In [10]:
base_mb = tabla1.query("archivo=='observations' and version=='inferido'")['MB'].iloc[0]

# 1) cardinalidad por columna
n = len(obs_naive)
for c in obs_naive.columns:
    print(f'{c:12s} nunique={obs_naive[c].nunique():>8}  ({obs_naive[c].nunique()/n:.1%} de {n})')

# 2) VALUE: fracción NO numérica y qué hay ahí
val_num = pd.to_numeric(obs_naive['VALUE'], errors='coerce')
print('\nVALUE no-numérico:', f'{val_num.isna().mean():.1%}')
print(obs_naive.loc[val_num.isna(),'VALUE'].value_counts().head(10))
print('\nTYPE:'); print(obs_naive['TYPE'].value_counts())

DATE         nunique= 1793984  (10.3% de 17355578)
PATIENT      nunique=   23018  (0.1% de 17355578)
ENCOUNTER    nunique=  732369  (4.2% de 17355578)
CATEGORY     nunique=       8  (0.0% de 17355578)
CODE         nunique=     298  (0.0% de 17355578)
DESCRIPTION  nunique=     300  (0.0% de 17355578)
VALUE        nunique=   45621  (0.3% de 17355578)
UNITS        nunique=      51  (0.0% de 17355578)
TYPE         nunique=       2  (0.0% de 17355578)

VALUE no-numérico: 36.9%
VALUE
No                                         1610881
Yes                                         401400
I have housing                              225943
Never smoked tobacco (finding)              222206
English                                     206371
White                                       185421
I choose not to answer this question        165953
Full-time work                              148081
Cloudy urine (finding)                      147319
Finding of bilirubin in urine (finding)     147015
Name: c

### Decisión de diseño para `VALUE`
Se evaluaron dos opciones:

1. Codificar todos los textos como enteros con leyenda en `UNITS` → **descartada**: la mayoría de los textos son nominales sin orden (`English`, `White`, `Yes/No`); la codificación tendría que ser por `DESCRIPTION` (298 codebooks); y `UNITS` no es un almacén de metadata.
2. **Partir `VALUE` en `VALUE_num` (float32) + `VALUE_cat` (category) y descartar el original.** ← elegida

La partición usa `TYPE` (ground truth de Synthea: `numeric`/`text`) para decidir fila por fila. Coincide con cómo OMOP CDM modela esto: `value_as_number` / `value_as_concept_id` en columnas separadas.

De esta manera ahorramos con las celdas de `VALUE` que eran numéricas con `float32` (alrededor de un 60% de las celdas) y con el resto con `category`.

### Implementación

In [11]:
def col_mb(df):
    return (df.memory_usage(deep=True)/1e6).round(3)  # MB por columna

antes = col_mb(obs_naive)

obs = obs_naive.copy()
obs['DATE'] = pd.to_datetime(obs['DATE'], utc=True)

for c in ['CATEGORY','CODE','DESCRIPTION','UNITS','TYPE']:
    obs[c] = obs[c].astype('category')

for c in ['PATIENT','ENCOUNTER']:
    obs[c] = obs[c].astype('category')   # UUID que repiten -> category (ver nota)

is_num = obs['TYPE'].eq('numeric')
obs['VALUE_num'] = pd.to_numeric(obs['VALUE'].where(is_num), errors='coerce').astype('float32')
obs['VALUE_cat'] = obs['VALUE'].where(~is_num).astype('category')
obs = obs.drop(columns='VALUE')

despues = col_mb(obs)
reduccion = 1 - mb(obs)/base_mb
print(f'Reducción total: {reduccion:.1%}')   # objetivo >=70%

Reducción total: 94.4%


Reducción total: **94.4%** (objetivo ≥ 70%, cumplido).

### Tabla por columna (entregable de la Actividad 2)
Las cinco primeras columnas se arman solas desde `antes`/`despues` y los dtypes. La última es que se pierde por el tipado.

In [12]:
notas = {   # qué se pierde en cada conversión
    'DATE':        'Nada sustantivo. La precisión de datetime64[ns] excede la del dato origen. '
                   'Se fija UTC: se pierde la zona horaria local si alguna vez importara.',
    'PATIENT':     'Ergonomía en los merge: category exige categorías compatibles entre tablas, '
                   'obliga a castear la clave antes de unir (ver Actividad 4).',
    'ENCOUNTER':   'Igual que PATIENT. Además el diccionario es grande (732 369 únicos), '
                   'por lo que el ahorro es menor que en PATIENT aunque sigue siendo alto.',
    'CATEGORY':    'Operaciones de texto vectorizadas (.str) sobre category son más lentas '
                   'o requieren volver a object.',
    'CODE':        'Igual que CATEGORY. Agregar un código nuevo obliga a extender las categorías.',
    'DESCRIPTION': 'Igual que CODE. Al ser etiqueta legible es la más propensa a operaciones '
                   'de texto, que es justo lo que category encarece.',
    'UNITS':       'Igual que CATEGORY.',
    'TYPE':        'Ninguna pérdida relevante: 2 valores únicos, cardinalidad mínima.',
    'VALUE_num':   'Precisión: float32 tiene ~7 dígitos significativos frente a ~16 de float64. '
                   'Verificado en la Actividad 7: diferencia máxima de 3.2e-4 en las medias por código.',
    'VALUE_cat':   'Se parte una columna en dos con NaN complementarios (38.4% / 61.6%): '
                   'hay que recordar cuál consultar según TYPE, y ninguna consulta cubre ambas a la vez.',
}

filas, visto_value = [], False
for col in obs.columns:
    en_orig = col in obs_naive.columns
    origen  = col if en_orig else 'VALUE'
    if en_orig:
        mb_antes, dt_antes = antes[col], str(obs_naive[col].dtype)
    elif not visto_value:                       # VALUE_num hereda el peso de VALUE
        mb_antes, dt_antes, visto_value = antes['VALUE'], 'object', True
    else:                                        # VALUE_cat: no re-contar
        mb_antes, dt_antes = 0.0, '(idem VALUE)'
    filas.append({'columna':col, 'dtype_antes':dt_antes, 'dtype_despues':str(obs[col].dtype),
                  'MB_antes':round(float(mb_antes),3), 'MB_despues':round(float(despues[col]),3),
                  'que_se_pierde':notas.get(col,'')})

tabla2 = pd.DataFrame(filas)
tabla2

,columna,dtype_antes,dtype_despues,MB_antes,MB_despues,que_se_pierde
0,DATE,object,"datetime64[ns, UTC]",1197.535,138.845,Nada sustantivo. La precisión de datetime64[ns...
1,PATIENT,object,category,1475.224,37.196,Ergonomía en los merge: category exige categor...
2,ENCOUNTER,object,category,1442.155,148.582,Igual que PATIENT. Además el diccionario es gr...
3,CATEGORY,object,category,989.334,17.356,Operaciones de texto vectorizadas (.str) sobre...
4,CODE,object,category,962.296,34.736,Igual que CATEGORY. Agregar un código nuevo ob...
5,DESCRIPTION,object,category,1594.169,34.749,Igual que CODE. Al ser etiqueta legible es la ...
6,UNITS,object,category,836.831,17.360,Igual que CATEGORY.
7,TYPE,object,category,951.934,17.356,"Ninguna pérdida relevante: 2 valores únicos, c..."
8,VALUE_num,object,float32,1004.791,69.422,Precisión: float32 tiene ~7 dígitos significat...
9,VALUE_cat,(idem VALUE),category,0.000,72.624,Se parte una columna en dos con NaN complement...


**Guía para `notas`:** `float32` pierde precisión decimal; `category` estorba operaciones de texto vectorizadas y complica los `merge` (Act. 4); partir `VALUE` obliga a llevar dos columnas con `NaN` complementarios.

**Nota `PATIENT`/`ENCOUNTER`:** son UUID que **repiten** (a diferencia de `patients.Id`, único), por eso `category` los aplasta (~20× frente a dejarlos como texto). El costo es ergonómico en los `merge`: se retoma en la Actividad 4.

## Actividad 3 — Auditoría de calidad

### Faltantes, duplicados y coherencia temporal

In [13]:
# (a) % faltantes por columna, en los tres
for name, df in [('patients',p8),('encounters',enc_naive),('observations',obs)]:
    print(f'== {name} ==\n{(df.isna().mean()*100).round(1)}\n')

# (b) Id de paciente duplicados
print('patients Id duplicados:', p8['Id'].duplicated().sum())

# (c) coherencia temporal
enc = enc_naive.copy()
enc['START'] = pd.to_datetime(enc['START'], utc=True)
ref = p8[['Id','BIRTHDATE','DEATHDATE']].rename(columns={'Id':'PATIENT'})
chk = enc.merge(ref, on='PATIENT', how='left')   # lookup; el merge RIGUROSO es la Act.4
chk['BIRTHDATE'] = pd.to_datetime(chk['BIRTHDATE'], utc=True)
chk['DEATHDATE'] = pd.to_datetime(chk['DEATHDATE'], utc=True)
antes_nacer = (chk['START'] < chk['BIRTHDATE']).sum()
tras_morir  = (chk['START'] > chk['DEATHDATE']).sum()
print('encuentros antes de nacer:', antes_nacer, '| tras defunción:', tras_morir)

== patients ==
Id            0.0
BIRTHDATE     0.0
DEATHDATE    86.9
RACE          0.0
ETHNICITY     0.0
GENDER        0.0
CITY          0.0
STATE         0.0
dtype: float64

== encounters ==
Id                0.0
START             0.0
STOP              0.0
PATIENT           0.0
ENCOUNTERCLASS    0.0
CODE              0.0
DESCRIPTION       0.0
dtype: float64

== observations ==
DATE            0.0
PATIENT         0.0
ENCOUNTER       3.6
CATEGORY        3.6
CODE            0.0
DESCRIPTION     0.0
UNITS          27.3
TYPE            0.0
VALUE_num      38.4
VALUE_cat      61.6
dtype: float64

patients Id duplicados: 0
encuentros antes de nacer: 0 | tras defunción: 3289


### Encuentros posteriores a la defunción
El chequeo marca **3 289** encuentros con `START > DEATHDATE`. Antes de reportarlo como anomalía, un matiz: `DEATHDATE` no tiene hora (queda a medianoche UTC), mientras que `START` es un timestamp completo. Un encuentro el mismo día de la muerte, a cualquier hora > 00:00, cae como 'posterior'. Estratificamos para separar el artefacto de la anomalía real:

In [14]:
mismo_dia = ((chk['START'].dt.normalize() == chk['DEATHDATE'].dt.normalize()) &
             (chk['START'] > chk['DEATHDATE'])).sum()
dias_desp = (chk['START'] > chk['DEATHDATE'] + pd.Timedelta(days=1)).sum()
print('mismo día (artefacto fecha vs timestamp):', mismo_dia)
print('más de 1 día después  :', dias_desp)

mismo día (artefacto fecha vs timestamp): 526
más de 1 día después  : 2763


La estratificación dio 526 el mismo día (artefacto fecha-vs-timestamp) y 2 763 a más de un día — o sea, la mayoría no es artefacto. Antes de concluir hay que ver qué son: cuántos días después, de qué clase, con qué descripción.

In [15]:
post = chk[chk['START'] > chk['DEATHDATE'] + pd.Timedelta(days=1)].copy()
post['dias_tras_muerte'] = (post['START'] - post['DEATHDATE']).dt.days
print(post['dias_tras_muerte'].describe())
print('\nPor clase de encuentro:')
print(post['ENCOUNTERCLASS'].value_counts())
print('\nDescripciones más frecuentes:')
print(post['DESCRIPTION'].value_counts().head(10))

count    2763.000000
mean        6.350706
std         3.764192
min         1.000000
25%         3.000000
50%         6.000000
75%         9.000000
max        14.000000
Name: dias_tras_muerte, dtype: float64

Por clase de encuentro:
ENCOUNTERCLASS
wellness    2763
Name: count, dtype: int64

Descripciones más frecuentes:
DESCRIPTION
Death Certification    2763
Name: count, dtype: int64


De los 3289 encuentros con START posterior a DEATHDATE: 526 son artefacto de
granularidad temporal (mismo día; la defunción se registra a medianoche y el
encuentro tiene hora). Los 2763 restantes son, sin excepción, encuentros
`wellness` de tipo "Death Certification", entre 1 y 14 días tras la muerte
(mediana 6). No son inconsistencias: la certificación ocurre necesariamente
después del fallecimiento.

### Checkpoint del estado curado
Se persisten los tres DataFrames ya tipados. El corte está aquí, y no al final, por un criterio de ingeniería de datos: **se guarda lo que es caro de calcular y estable**, y se recalcula lo que es barato de calcular o caro de almacenar.

- `obs`, `p8` y `enc_tip` son caros (lectura de 17.4 M filas más todas las conversiones de la Actividad 2) y estables: las actividades 4 a 9 los consumen sin modificarlos.
- `m1` y `m2` se descartan del checkpoint: son un merge de segundos, pero ocupan ~2 GB y escribirlos a Drive tardaría más que rehacerlos.

Es la misma frontera que separa la capa curada de la capa de análisis en un pipeline de producción.

In [16]:
CKPT = BASE / 'ckpt'; CKPT.mkdir(exist_ok=True)
obs.to_parquet(CKPT/'obs.parquet', index=False)
p8.to_parquet(CKPT/'p8.parquet', index=False)
enc_tip.to_parquet(CKPT/'enc_tip.parquet', index=False)
print('checkpoint guardado')

checkpoint guardado


## Actividad 4 — Unir las tres tablas validando cardinalidades

Unimos `observations` → `encounters` → `patients`. El punto no es lograr el join, sino **declarar la cardinalidad esperada antes** y que pandas la verifique.

Dos parámetros nuevos:
- `validate='m:1'` — verifica que la clave del lado **derecho** sea única (many-to-one). Si no lo es, pandas levanta `MergeError` en vez de multiplicar filas en silencio.
- `indicator=True` — agrega una columna `_merge` con `left_only` / `right_only` / `both`, para auditar qué se unió y qué no.

### Cardinalidad esperada (antes de tocar código)

**Join 1 — `observations` → `encounters`** (`obs.ENCOUNTER` = `enc.Id`)
- Cada observación pertenece a **un** encuentro; cada encuentro tiene **muchas** observaciones → **many-to-one** (`validate='m:1'`).
- Filas esperadas: **sin cambio** (≈ 17.36 M), porque con `how='left'` cada observación trae ≤ 1 encuentro.
- Salvedad de la auditoría: **3.6% de `observations` tiene `ENCOUNTER` nulo** → esas filas quedan `left_only`. Es esperado, no un error.

**Join 2 — resultado → `patients`** (`PATIENT` = `patients.Id`)
- Cada fila pertenece a **un** paciente; cada paciente tiene **muchas** filas → **many-to-one** (`validate='m:1'`).
- Filas esperadas: **sin cambio**. `patients.Id` es único (0 duplicados, auditado) → no hay fan-out.
- Esperamos **100% `both`**: toda observación pertenece a un paciente generado.

### Preparación de claves — el costo ergonómico de `category`
 `PATIENT` y `ENCOUNTER` están en `category` (óptimo en memoria), pero un merge sobre `category` exige categorías idénticas en ambos lados o pandas upcastea con warning. Para el join llevamos las claves a un tipo común: `string[pyarrow]` — Arrow-backed, merge limpio y más liviano que `object`. (Si diera problemas de versión, `.astype(str)` siempre funciona.)

> **RAM:** las claves son UUID de alta cardinalidad; materializarlas para el join pesa (~700 MB por columna sobre 17 M filas). Es inherente a unir por UUID: `category` ahorró memoria *en reposo*, pero el join tiene que comparar los valores reales. Llevamos solo las columnas necesarias de cada tabla para no inflar el resultado.

In [17]:
KEY = 'string[pyarrow]'   # tipo común para las claves de join

obs_m = obs.copy()
obs_m['ENCOUNTER'] = obs_m['ENCOUNTER'].astype(KEY)
obs_m['PATIENT']   = obs_m['PATIENT'].astype(KEY)

# encounters: solo lo necesario; Id es la clave derecha del join 1
enc_m = enc_tip[['Id','PATIENT','START','STOP','ENCOUNTERCLASS']].copy()
enc_m['Id']      = enc_m['Id'].astype(KEY)
enc_m['PATIENT'] = enc_m['PATIENT'].astype(KEY)

# patients: renombrar Id -> PATIENT para la clave del join 2
pat_m = p8.rename(columns={'Id':'PATIENT'}).copy()
pat_m['PATIENT'] = pat_m['PATIENT'].astype(KEY)

print('enc.Id duplicados     :', enc_m['Id'].duplicated().sum())        # 0 => ok para m:1
print('pat.PATIENT duplicados:', pat_m['PATIENT'].duplicated().sum())   # 0 => ok para m:1
print('filas observations    :', len(obs_m))

enc.Id duplicados     : 0
pat.PATIENT duplicados: 0
filas observations    : 17355578


### Join 1 — observations → encounters

In [18]:
m1 = obs_m.merge(
    enc_m, left_on='ENCOUNTER', right_on='Id', how='left',
    validate='m:1', indicator=True, suffixes=('', '_enc')
)
print('filas m1 (después):', len(m1))
print(m1['_merge'].value_counts())
m1 = m1.rename(columns={'_merge': '_merge_enc'})   # liberar el nombre para el 2do join

filas m1 (después): 17355578
_merge
both          16731626
left_only       623952
right_only           0
Name: count, dtype: int64


El join respetó la cardinalidad m:1: 17.36 M filas antes y después, sin
multiplicación. Las 623 952 `left_only` (3.6%) son las observaciones con
ENCOUNTER nulo ya detectadas en la auditoría — no matchean porque no tienen
clave, no porque falte el encuentro.




### Chequeo de integridad referencial
Tras el join, hay dos columnas de paciente: `PATIENT` (de observations) y `PATIENT_enc` (del encuentro). Donde hubo match deben coincidir — si no, una observación estaría atribuida a un encuentro de **otro** paciente.

In [19]:
both = m1['_merge_enc'].eq('both')
mismatch = (m1.loc[both, 'PATIENT'] != m1.loc[both, 'PATIENT_enc']).sum()
print('PATIENT inconsistente (obs vs enc):', mismatch)   # esperado 0

m1 = m1.drop(columns=['Id', 'PATIENT_enc'])   # sobran: clave duplicada y PATIENT del encuentro

PATIENT inconsistente (obs vs enc): 0


### Join 2 — (observations+encounters) → patients

In [20]:
m2 = m1.merge(
    pat_m, on='PATIENT', how='left',
    validate='m:1', indicator=True
)
print('filas m2 (después):', len(m2))
print(m2['_merge'].value_counts())

filas m2 (después): 17355578
_merge
both          17355578
left_only            0
right_only           0
Name: count, dtype: int64


Cualquier left_only sería una observación sin paciente en la tabla, pero aquí se observan 100% both. A Synthea no se le ha escapado ninguna.

### Reconciliación de conteos (entregable)


In [21]:
print('observations (inicio):', len(obs))
print('tras join encounters :', len(m1))
print('tras join patients   :', len(m2))
print('\n% match encuentros:', f"{m1['_merge_enc'].eq('both').mean():.1%}")
print('% match pacientes  :', f"{m2['_merge'].eq('both').mean():.1%}")

observations (inicio): 17355578
tras join encounters : 17355578
tras join patients   : 17355578

% match encuentros: 96.4%
% match pacientes  : 100.0%


Las filas se mantuvieron en 17 M en los dos joins (sin multiplicación), lo que confirma las cardinalidades `m:1` declaradas. El único 'faltante' es el `left_only` de encuentros, ya explicado en la auditoría (`ENCOUNTER` nulo). Si `validate` hubiera levantado `MergeError`, ahí estaría *el bug más caro del curso*: una clave que se creía única y no lo era, multiplicando filas en silencio.

## Actividad 5 — Preguntas clínicas

Cuatro preguntas sobre el dataset. **No usamos `m2`**: cada una se responde sobre la tabla mínima (`p8`, `enc_tip`, `obs`), lo que evita el costo de RAM de la tabla unida y deja el código más claro.

> Gotcha de `category`: todos los `groupby` sobre columnas categóricas van con **`observed=True`**. Sin eso, pandas genera un grupo por **cada** categoría existente (p. ej. los 23 018 pacientes) aunque el subconjunto filtrado tenga unos pocos — desperdicia memoria y ensucia el resultado.

### 1. Pacientes por grupo étnico y sexo

Decisión de agrupamiento: en el estándar de EE. UU. (OMB / censo), **raza y etnia son ejes ortogonales** — "hispano" es una etnia, no una raza. Por eso reportamos `RACE × GENDER` y `ETHNICITY × GENDER` por separado, en vez de mezclarlos en una sola variable. Es también como OMOP modela la demografía (`race_concept_id` y `ethnicity_concept_id` separados).

In [22]:
# Raza x sexo
raza_sexo = (p8.groupby(['RACE','GENDER'], observed=True)
               .size().unstack(fill_value=0))
print(raza_sexo, '\n')

# Etnia (hispano / no hispano) x sexo — eje ortogonal a la raza
etnia_sexo = (p8.groupby(['ETHNICITY','GENDER'], observed=True)
                .size().unstack(fill_value=0))
print(etnia_sexo)

GENDER       F     M
RACE                
asian      708   802
black      921   994
hawaiian   162   124
native      63    62
other      136   124
white     9561  9361 

GENDER           F      M
ETHNICITY                
hispanic      1303   1272
nonhispanic  10248  10195


**Justificación de los grupos.** Se reportan `RACE × GENDER` y `ETHNICITY × GENDER` **por separado**, no combinados en una sola variable. El motivo es que en el estándar estadounidense (OMB / censo) **raza y etnia son ejes ortogonales**: "hispano" es una etnia que puede coexistir con cualquier raza. Colapsarlas en una variable única perdería esa información y produciría categorías no comparables con las fuentes oficiales. Es también la forma en que OMOP CDM modela la demografía, con `race_concept_id` y `ethnicity_concept_id` en campos distintos.

Las categorías minoritarias (`native`, n=125; `other`, n=260; `hawaiian`, n=286) se **conservan sin colapsar**: agruparlas en "otras" ganaría estabilidad estadística pero borraría poblaciones que suelen ser justamente las de interés en estudios de equidad en salud. Se declara explícitamente que sus tamaños muestrales son pequeños y que cualquier estimación sobre ellas tendrá intervalos amplios.

### 2. Encuentros por paciente: media y mediana
Contamos encuentros por paciente y resumimos. Denominador: pacientes **con al menos un encuentro** (los que aparecen en `encounters`).

In [23]:
enc_por_pac = enc_tip.groupby('PATIENT', observed=True).size()

print('media  :', round(enc_por_pac.mean(), 2))
print('mediana:', enc_por_pac.median())
print()
print(enc_por_pac.describe())

media  : 58.9
mediana: 36.0

count    23018.000000
mean        58.900643
std         90.961079
min          1.000000
25%         24.000000
50%         36.000000
75%         57.000000
max        916.000000
dtype: float64


La media suele quedar **por encima** de la mediana: la distribución de encuentros es asimétrica a la derecha (pocos pacientes crónicos con muchísimos encuentros estiran la cola). Para "el paciente típico", la **mediana** representa mejor; la media es sensible a esos outliers.

### 3. Los 10 códigos de observación más frecuentes
`value_counts` sobre `CODE`, y le pegamos la descripción legible (una por código).

In [24]:
top10 = obs['CODE'].value_counts().head(10)

# descripción por código: primera aparición de cada CODE (pareado casi 1:1)
desc = obs.drop_duplicates('CODE').set_index('CODE')['DESCRIPTION']

tbl = top10.rename('frecuencia').to_frame()
tbl['descripcion'] = desc.reindex(tbl.index).astype(str).values
tbl

,frecuencia,descripcion
CODE,,
72514-3,566572,Pain severity - 0-10 verbal numeric rating [Sc...
8480-6,329381,Systolic Blood Pressure
8462-4,329381,Diastolic Blood Pressure
29463-7,313708,Body Weight
8867-4,306907,Heart rate
9279-1,306907,Respiratory rate
8302-2,300653,Body Height
72166-2,299901,Tobacco smoking status
39156-5,278560,Body mass index (BMI) [Ratio]


### 4. Analito elegido: BMI (Body mass index)
Distinto de los ejemplos del README y lo utilizo en mi proyecto de doctorado.

In [25]:
BMI_CODE = '39156-5'                    # LOINC: BMI [Ratio]
bmi = obs[obs['CODE'] == BMI_CODE]
print('mediciones de BMI:', len(bmi))

mediciones de BMI: 278560


#### Distribución y pacientes con ≥ 3 mediciones

In [26]:
vals = bmi['VALUE_num'].dropna()
print(vals.describe(), '\n')

n_por_pac = bmi.groupby('PATIENT', observed=True)['VALUE_num'].count()
print('pacientes con >=3 mediciones de BMI:', (n_por_pac >= 3).sum())

count    278560.000000
mean         26.728586
std           4.666247
min           7.900000
25%          26.600000
50%          27.900000
75%          29.600000
max          55.000000
Name: VALUE_num, dtype: float64 

pacientes con >=3 mediciones de BMI: 22178


#### La media de medias (punto de la actividad)
Dos formas de "promediar" el BMI que **no** dan lo mismo cuando cada paciente tiene distinto número de mediciones:

In [27]:
media_global   = vals.mean()                                   # todas las mediciones pesan igual
media_por_pac  = bmi.groupby('PATIENT', observed=True)['VALUE_num'].mean()
media_de_medias = media_por_pac.mean()                          # cada paciente pesa igual

print('media global (todas las mediciones):', round(media_global, 2))
print('media de medias por paciente       :', round(media_de_medias, 2))

media global (todas las mediciones): 26.73
media de medias por paciente       : 25.94


**Media global vs. media de medias.** Las dos cifras difieren porque ponderan distinto: en la media global cada **medición** pesa igual, de modo que un paciente con 20 registros de IMC influye 20 veces más que uno con un solo registro; en la media de medias cada **paciente** pesa igual, sin importar cuántas veces se lo midió.

La pregunta que se responde acá es de tipo poblacional — *cuál es el IMC del paciente típico de esta cohorte* —, y por lo tanto la respuesta correcta es la **media de medias**: cada persona debe contar una vez. Usar la media global sesgaría el resultado hacia los pacientes con más contactos con el sistema de salud, que son sistemáticamente los más enfermos y los que más mediciones acumulan. Ese sesgo de intensidad de seguimiento es un problema clásico de los datos de EHR y no un detalle de cálculo.

La media global sería la respuesta adecuada a otra pregunta: *cuánto vale una medición típica de IMC en esta base*, por ejemplo para calibrar un instrumento o dimensionar un rango de referencia.

## Actividad 6 — Formato ancho y valores implausibles

Reestructuramos las observaciones a **formato ancho** (un renglón por paciente-fecha, una columna por analito) y detectamos valores fuera de rango fisiológico.

> **Sobre la pista del enunciado:** menciona `melt` y `pivot_table`. `melt` convierte *ancho → largo*, y `observations` **ya está en formato largo** (una fila por medición, con `DESCRIPTION` como analito y `VALUE_num` como valor). Aplicar `melt` sería un paso innecesario. Solo hace falta `pivot_table`.

### 6.1 Por qué no pivotamos todos los analitos
`DESCRIPTION` tiene ~300 valores. Pivotar todos generaría una tabla de (pares paciente-fecha) × 300 columnas, casi toda `NaN`: ningún paciente se mide 300 analitos el mismo día. Es una explosión de memoria sin información añadida.

El formato ancho solo es viable sobre un **subconjunto curado** de analitos. Elegimos los más frecuentes entre los numéricos (signos vitales y laboratorios comunes).

In [28]:
num = obs[obs['VALUE_num'].notna()]
top_analitos = num['DESCRIPTION'].value_counts().head(12)
print(top_analitos)

ANALITOS = top_analitos.index.tolist()   # ajustá esta lista si querés otros

DESCRIPTION
Pain severity - 0-10 verbal numeric rating [Score] - Reported                                                    566572
Systolic Blood Pressure                                                                                          329381
Diastolic Blood Pressure                                                                                         329381
Body Weight                                                                                                      313708
Respiratory rate                                                                                                 306907
Heart rate                                                                                                       306907
Body Height                                                                                                      300653
Body mass index (BMI) [Ratio]                                                                                    278560
Glomerular filtration rate [

### 6.2 Normalizar la fecha al día
`DATE` es un timestamp completo. El enunciado pide un renglón por paciente y **fecha**, así que truncamos al día con `dt.normalize()`.

Esto **es** lo que genera las colisiones que el enunciado advierte: dos mediciones del mismo analito en el mismo día caen en la misma celda. Con el timestamp completo casi no habría colisiones y el problema quedaría escondido.

In [29]:
sub = obs[obs['DESCRIPTION'].isin(ANALITOS) & obs['VALUE_num'].notna()].copy()
sub['FECHA'] = sub['DATE'].dt.normalize()
print('mediciones seleccionadas:', len(sub))
print('pares paciente-fecha únicos:', sub[['PATIENT','FECHA']].drop_duplicates().shape[0])

mediciones seleccionadas: 3635074
pares paciente-fecha únicos: 864875


### 6.3 Medir las colisiones ANTES de pivotar
`pivot_table` agrega en silencio: si hay dos valores para la misma celda, aplica `aggfunc` sin avisar. Así que primero contamos cuántas celdas tienen más de un valor — recién con ese número se puede elegir la agregación con criterio.

In [30]:
conteo = sub.groupby(['PATIENT','FECHA','DESCRIPTION'], observed=True).size()

print('celdas (paciente-fecha-analito):', len(conteo))
print('celdas con >1 medición        :', (conteo > 1).sum(), f'({(conteo > 1).mean():.2%})')
print('\nmediciones por celda:')
print(conteo.value_counts().sort_index().head())

celdas (paciente-fecha-analito): 3596237
celdas con >1 medición        : 38731 (1.08%)

mediciones por celda:
1    3557506
2      38625
3        106
Name: count, dtype: int64


**Decisión de agregación.** Las colisiones son marginales: 38 731 celdas sobre 3 596 237 (**1.08%**), y casi todas con solo dos mediciones (38 625 de dos, 106 de tres). Con esa proporción, cualquier `aggfunc` mueve poco el resultado global.

Se elige **`'max'`** porque uno de los objetivos de la actividad es detectar valores fuera de rango fisiológico, y conservar el extremo superior evita que un promedio diluya un valor anómalo alto.

**Lo que esta elección cuesta, y es importante declararlo:** `'max'` protege el extremo *alto* y sacrifica el *bajo*. Si un paciente tiene dos mediciones el mismo día y una es implausiblemente baja, `'max'` la descarta y el chequeo posterior no la ve. Dado que los hallazgos de esta actividad resultaron estar precisamente en el extremo inferior (presión sistólica e IMC bajos), la elección es conservadora en la dirección equivocada: el conteo de implausibles reportado más abajo es un **piso**, no un total. Con `'min'` el sesgo sería el inverso; una alternativa neutral habría sido conservar ambos extremos en dos tablas anchas separadas.

### 6.4 El pivote
Dos parámetros que no son opcionales acá:
- `aggfunc=` — explícito, según lo decidido arriba. El default de `pivot_table` es `'mean'`; pero por lo explicado arriba elegimos `'max'`.
- `observed=True` — `DESCRIPTION` es `category`, y sin esto pandas genera una columna por **cada una de las ~300 categorías**, no solo por las 12 filtradas.

In [31]:
ancho = sub.pivot_table(
    index=['PATIENT','FECHA'],
    columns='DESCRIPTION',
    values='VALUE_num',
    aggfunc='max',
    observed=True,
)

print('shape:', ancho.shape)
print('memoria:', round(mb(ancho), 1), 'MB')
ancho.head()

shape: (864875, 12)
memoria: 47.7 MB


DESCRIPTION                                                     Body Height  \
PATIENT                              FECHA                                    
00036c56-eb1b-d0eb-e759-ce835e3ef7d1 2016-09-19 00:00:00+00:00   161.399994   
                                     2016-10-09 00:00:00+00:00          NaN   
                                     2017-09-25 00:00:00+00:00   161.399994   
                                     2017-10-09 00:00:00+00:00          NaN   
                                     2018-10-01 00:00:00+00:00   161.399994   

DESCRIPTION                                                     Body Weight  \
PATIENT                              FECHA                                    
00036c56-eb1b-d0eb-e759-ce835e3ef7d1 2016-09-19 00:00:00+00:00    78.099998   
                                     2016-10-09 00:00:00+00:00          NaN   
                                     2017-09-25 00:00:00+00:00    78.500000   
                                     2017-10-09 00:00:00+00:00          NaN   
                                     2018-10-01 00:00:00+00:00    78.500000   

DESCRIPTION                                                     Body mass index (BMI) [Ratio]  \
PATIENT                              FECHA                                                      
00036c56-eb1b-d0eb-e759-ce835e3ef7d1 2016-09-19 00:00:00+00:00                           30.0   
                                     2016-10-09 00:00:00+00:00                            NaN   
                                     2017-09-25 00:00:00+00:00                           30.1   
                                     2017-10-09 00:00:00+00:00                            NaN   
                                     2018-10-01 00:00:00+00:00                           30.1   

DESCRIPTION                                                     DALY  \
PATIENT                              FECHA                             
00036c56-eb1b-d0eb-e759-ce835e3ef7d1 2016-09-19 00:00:00+00:00   NaN   
                                     2016-10-09 00:00:00+00:00   1.6   
                                     2017-09-25 00:00:00+00:00   NaN   
                                     2017-10-09 00:00:00+00:00   1.7   
                                     2018-10-01 00:00:00+00:00   NaN   

DESCRIPTION                                                     Diastolic Blood Pressure  \
PATIENT                              FECHA                                                 
00036c56-eb1b-d0eb-e759-ce835e3ef7d1 2016-09-19 00:00:00+00:00                      65.0   
                                     2016-10-09 00:00:00+00:00                       NaN   
                                     2017-09-25 00:00:00+00:00                      73.0   
                                     2017-10-09 00:00:00+00:00                       NaN   
                                     2018-10-01 00:00:00+00:00                      69.0   

DESCRIPTION                                                     Glomerular filtration rate [Volume Rate/Area] in Serum or Plasma by Creatinine-based formula (MDRD)/1.73 sq M  \
PATIENT                              FECHA                                                                                                                                      
00036c56-eb1b-d0eb-e759-ce835e3ef7d1 2016-09-19 00:00:00+00:00                                                NaN                                                               
                                     2016-10-09 00:00:00+00:00                                                NaN                                                               
                                     2017-09-25 00:00:00+00:00                                                NaN                                                               
                                     2017-10-09 00:00:00+00:00                                                NaN                                                               


In [32]:
# Densidad: qué proporción de celdas tiene dato (el formato ancho es intrínsecamente disperso)
print('celdas con valor:', f'{ancho.notna().to_numpy().mean():.1%}')
print('\nvalores no nulos por analito:')
print(ancho.notna().sum().sort_values(ascending=False))

celdas con valor: 34.7%

valores no nulos por analito:
DESCRIPTION
Pain severity - 0-10 verbal numeric rating [Score] - Reported                                                    566474
Diastolic Blood Pressure                                                                                         329046
Systolic Blood Pressure                                                                                          329046
Body Weight                                                                                                      312871
Respiratory rate                                                                                                 306572
Heart rate                                                                                                       306572
Body Height                                                                                                      300653
Body mass index (BMI) [Ratio]                                                                

### 6.5 Rangos fisiológicos — **implausible** ≠ **anormal**
Distinción central antes de fijar umbrales:
- **Anormal**: fuera del rango de referencia clínico, pero biológicamente posible. Un IMC de 45 es anormal y perfectamente real.
- **Implausible**: biológicamente imposible o producto de un error de registro. Un IMC de 900, una temperatura de 200 °C, un peso negativo.

Esta actividad pide detectar lo **segundo**. Marcar como error todo lo clínicamente anormal eliminaría justamente a los pacientes enfermos, que es el sesgo más costoso que se puede introducir en datos clínicos.

#### Fuentes de los rangos
Los límites no se inventan. Se usan tres orígenes, declarados por analito en el código:

- **[A] Rangos publicados de plausibilidad fisiológica para signos vitales.** Tabla 4 ("Physiologically Plausible Range") de: *Improving Clinical Decision Support through Interpretable Machine Learning and Error Handling in Electronic Health Records*, arXiv:2308.10781. Aporta frecuencia cardíaca (30–200), respiratoria (8–70), presión sistólica (50–200) y diastólica (20–150).
- **[B] Rango definicional del instrumento.** La escala verbal numérica de dolor está definida entre 0 y 10; cualquier valor fuera de ese intervalo es un error de registro por construcción.
- **[C] Límites convencionales de control de calidad antropométrico** (talla, peso, IMC). **Limitación declarada:** no se apoyan en una fuente publicada específica y deben tomarse como umbrales operativos, no como valores de referencia citables.

**Marco metodológico:** la categoría de *plausibility* utilizada acá proviene de Kahn MG, Callahan TJ, Barnard J, et al. (2016), *A Harmonized Data Quality Assessment Terminology and Framework for the Secondary Use of Electronic Health Record Data*, EGEMS 4(1):1244. Es el marco que implementa el **OHDSI Data Quality Dashboard** mediante sus chequeos `plausibleValueLow` / `plausibleValueHigh` (https://ohdsi.github.io/DataQualityDashboard/).

> **Nota relevante para interpretar los resultados.** El propio proyecto OHDSI retiró la mayoría de los rangos `plausibleValueLow`/`plausibleValueHigh` de sus archivos de umbrales a nivel concepto, porque la comunidad reportó que esos intervalos estaban marcando como fallas valores que en realidad eran plausibles. Es exactamente la tensión que aparece más abajo con las presiones sistólicas bajas: fijar un umbral de plausibilidad demasiado estrecho convierte casos clínicos extremos pero reales en falsos positivos.

**Analitos excluidos del chequeo:** `DALY`, `Glomerular filtration rate (MDRD)`, `Weight difference pre/post diálisis` y `How many people are living or staying at this address` no reciben rango. Los dos primeros son índices derivados y los dos últimos no son mediciones fisiológicas; en ningún caso se dispone de un límite de plausibilidad citable, y asignarles uno arbitrario sería precisamente el error que esta sección busca evitar.

In [33]:
# Límites de PLAUSIBILIDAD (no de normalidad clínica).
# Las claves coinciden EXACTAMENTE con los nombres de columna de `ancho`.
# Fuente por analito: [A] arXiv:2308.10781 Tabla 4 | [B] definición del instrumento
#                     [C] convencional de QC (limitación declarada en el texto)
RANGOS = {
    # [A] signos vitales, rangos publicados de plausibilidad fisiológica
    'Heart rate':                                                    (30, 200),   # lpm
    'Respiratory rate':                                              (8, 70),     # rpm
    'Systolic Blood Pressure':                                       (50, 200),   # mmHg
    'Diastolic Blood Pressure':                                      (20, 150),   # mmHg
    # [B] rango definicional del instrumento
    'Pain severity - 0-10 verbal numeric rating [Score] - Reported': (0, 10),     # score
    # [C] antropometría, umbrales convencionales de QC
    'Body Height':                                                   (30, 250),   # cm
    'Body Weight':                                                   (0.5, 500),  # kg
    'Body mass index (BMI) [Ratio]':                                 (10, 70),    # kg/m2
}

FUENTE = {
    'Heart rate':'[A]', 'Respiratory rate':'[A]',
    'Systolic Blood Pressure':'[A]', 'Diastolic Blood Pressure':'[A]',
    'Pain severity - 0-10 verbal numeric rating [Score] - Reported':'[B]',
    'Body Height':'[C]', 'Body Weight':'[C]', 'Body mass index (BMI) [Ratio]':'[C]',
}

# Control: ninguna clave debe quedar sin columna (si queda, se saltearía en silencio)
sin_match = [k for k in RANGOS if k not in ancho.columns]
sin_rango = [c for c in ancho.columns if c not in RANGOS]
print('claves de RANGOS sin columna (debe estar vacío):', sin_match)
print('\ncolumnas de `ancho` sin rango (exclusión deliberada, ver texto):')
for c in sin_rango: print(' -', c)

claves de RANGOS sin columna (debe estar vacío): []

columnas de `ancho` sin rango (exclusión deliberada, ver texto):
 - DALY
 - Glomerular filtration rate [Volume Rate/Area] in Serum or Plasma by Creatinine-based formula (MDRD)/1.73 sq M
 - How many people are living or staying at this address [#]
 - Weight difference [Mass difference] --pre dialysis - post dialysis


### 6.6 Conteo de implausibles

In [34]:
filas = []
for col, (lo, hi) in RANGOS.items():
    if col not in ancho.columns:
        continue
    s = ancho[col]
    fuera = ((s < lo) | (s > hi))
    filas.append({
        'analito': col, 'rango': f'[{lo}, {hi}]', 'fuente': FUENTE.get(col,''),
        'n_valores': int(s.notna().sum()),
        'implausibles': int(fuera.sum()),
        'pct': round(100 * fuera.sum() / max(s.notna().sum(), 1), 3),
        'min_obs': round(float(s.min()), 2) if s.notna().any() else None,
        'max_obs': round(float(s.max()), 2) if s.notna().any() else None,
    })

tabla6 = pd.DataFrame(filas)
tabla6

,analito,rango,fuente,n_valores,implausibles,pct,min_obs,max_obs
0,Heart rate,"[30, 200]",[A],306572,0,0.000,50.0,200.0
1,Respiratory rate,"[8, 70]",[A],306572,0,0.000,12.0,40.0
2,Systolic Blood Pressure,"[50, 200]",[A],329046,41,0.012,40.1,188.0
3,Diastolic Blood Pressure,"[20, 150]",[A],329046,0,0.000,26.0,134.0
4,Pain severity - 0-10 verbal numeric rating [Sc...,"[0, 10]",[B],566474,0,0.000,0.0,10.0
5,Body Height,"[30, 250]",[C],300653,0,0.000,45.1,195.1
6,Body Weight,"[0.5, 500]",[C],312871,0,0.000,1.8,164.5
7,Body mass index (BMI) [Ratio],"[10, 70]",[C],278560,4,0.001,7.9,55.0


**Hallazgos.** Con los rangos citados, el chequeo sí marca valores: los extremos inferiores de **presión sistólica** (mínimo observado 40.1 mmHg, frente al límite de 50 de la fuente [A]) y de **IMC** (mínimo observado 7.9 kg/m², frente al límite de 10). El resto de los analitos queda íntegramente dentro de rango.

**Interpretación, y es la parte que importa.** Estos valores probablemente **no sean errores de registro**. Una presión sistólica de 40 mmHg es shock profundo: una situación clínica real, grave y perfectamente registrable. Un IMC de 7.9 corresponde a desnutrición extrema, también documentada. Es decir, el chequeo está capturando **casos clínicos extremos pero plausibles**, no artefactos — exactamente el problema que llevó al proyecto OHDSI a retirar la mayoría de sus rangos de plausibilidad por generar falsos positivos.

La conclusión metodológica es que un umbral de plausibilidad demasiado estrecho no limpia los datos: **elimina a los pacientes más graves**, que suelen ser la población de interés. Por eso el criterio adoptado en 6.7 es marcar y documentar, nunca descartar en silencio.

**Sobre los datos sintéticos.** Que el conteo sea tan bajo es consecuencia de trabajar con Synthea, que genera valores dentro de rangos fisiológicos por construcción. Lo que se entrega acá es el **procedimiento de control**, no el hallazgo. Sobre un EHR real este mismo chequeo detectaría errores de unidad (peso en libras cargado como kilogramos), errores de tipeo (un dígito de más) y valores centinela (999, -1 usados como "desconocido").

**Recordatorio del sesgo introducido en 6.3:** la agregación por `'max'` descarta la menor de dos mediciones del mismo día, de modo que algunos valores bajos implausibles quedaron ocultos antes de llegar a este chequeo. El conteo reportado es un piso.

### 6.7 Qué hacer con los valores implausibles
Tres políticas, de menos a más destructiva:

1. **Marcar sin borrar** — columna booleana de auditoría; el valor original queda. Reversible, trazable.
2. **Anular el valor** (`NaN`) conservando la fila — se pierde ese dato, se conservan los demás analitos de esa fila.
3. **Eliminar la fila** — la más destructiva: descarta mediciones válidas de otros analitos por un solo valor malo.

Por lo tanto considero que las opciones válidas son la primera o la segunda, dependiendo la fuente de los datos con la que estemos trabajando.

### Checkpoint
La tabla ancha es moderadamente cara de construir y estable de acá en adelante: la persistimos.

In [35]:
ancho.to_parquet(CKPT/'ancho.parquet')
print('guardado:', CKPT/'ancho.parquet')

guardado: /content/drive/MyDrive/curso-cd-2027/lab02/ckpt/ancho.parquet


## Actividad 7 — Procesar por lotes lo que no cabe

Se fija un **presupuesto de memoria de 200 MB** y se procesa `observations.csv` **completo** (17.36 M filas) sin superarlo, calculando media y conteo por código de observación.

La lógica se invierte respecto de las actividades anteriores: en vez de optimizar el DataFrame ya cargado, no se carga nunca entero.

### Por qué el enfoque respeta el presupuesto
Tres propiedades del diseño:

1. **`chunksize` devuelve un iterador**: `read_csv` entrega bloques de N filas. En memoria vive **un bloque a la vez**; al pedir el siguiente, el anterior queda libre.
2. **El acumulador es O(códigos únicos), no O(filas)**: guarda suma y conteo por cada uno de los ~300 códigos. No crece con el volumen de datos — procesar 17 M o 170 M filas da el mismo acumulador.
3. **Media por agregación aditiva**: la media global no necesita todos los valores a la vez. Se acumula `suma` y `n` por grupo y se divide al final.

El pico de memoria queda determinado por el tamaño del bloque, que es un parámetro que controlamos.

### Cómo se mide (y por qué de esta forma)
Dos medidores, que no miden lo mismo:

- **`tracemalloc`** (biblioteca estándar) rastrea asignaciones del *allocator de Python*. Es preciso para objetos Python, pero **puede no capturar los buffers que NumPy reserva directamente del sistema** — es decir, justo donde vive el grueso de un DataFrame.
- **RSS vía `memory_profiler`** mide la memoria física del proceso completo. Captura todo, incluido NumPy.

Se reporta **RSS incremental** (pico durante el procesamiento menos la línea base), porque el RSS absoluto incluye el intérprete, pandas y todo lo ya cargado — el presupuesto se refiere a lo que agrega el procesamiento, no al proceso entero.

In [36]:
import time, gc, os, tracemalloc
import psutil
from memory_profiler import memory_usage

proc = psutil.Process(os.getpid())
def rss_mib():
    return proc.memory_info().rss / (1024**2)

def medir(fn):
    """Ejecuta fn() y devuelve (resultado, segundos, pico RSS incremental en MiB)."""
    gc.collect()
    base = rss_mib()
    caja = {}
    def _run():
        caja['res'] = fn()
    t0 = time.perf_counter()
    pico = memory_usage((_run, (), {}), max_usage=True, interval=0.05)
    seg = time.perf_counter() - t0
    return caja['res'], seg, pico - base

### Implementación por lotes

In [37]:
PRESUPUESTO_MIB = 200
CHUNK = 250_000          # filas por bloque; es la perilla que controla el pico

def agregar_por_codigo(path, chunksize=CHUNK):
    """Media y conteo de VALUE por CODE, recorriendo el CSV por bloques."""
    total = None
    for bloque in pd.read_csv(path, usecols=['CODE','VALUE','TYPE'], chunksize=chunksize):
        # solo los valores numéricos (TYPE es el ground truth de Synthea)
        v = pd.to_numeric(bloque['VALUE'].where(bloque['TYPE'].eq('numeric')),
                          errors='coerce')
        g = v.groupby(bloque['CODE'], observed=True).agg(['sum','count'])
        # acumulación aditiva; add() alinea índices y rellena códigos nuevos
        total = g if total is None else total.add(g, fill_value=0)
    total['media'] = total['sum'] / total['count']
    return total

res7, seg7, pico7 = medir(lambda: agregar_por_codigo(DATA/'observations.csv'))

print(f'tiempo          : {seg7:.1f} s')
print(f'pico incremental: {pico7:.1f} MiB   (presupuesto {PRESUPUESTO_MIB} MiB)')
print(f'¿dentro del presupuesto?: {pico7 < PRESUPUESTO_MIB}')
print(f'códigos agregados: {len(res7)}')
res7.sort_values('count', ascending=False).head(10)

tiempo          : 52.9 s
pico incremental: 4.6 MiB   (presupuesto 200 MiB)
¿dentro del presupuesto?: True
códigos agregados: 298


,sum,count,media
CODE,,,
72514-3,1758449.0,566572.0,3.103664
8462-4,25846789.8,329381.0,78.470798
8480-6,38557643.1,329381.0,117.060921
29463-7,22038543.6,313708.0,70.251774
8867-4,24838525.1,306907.0,80.931765
9279-1,4372067.1,306907.0,14.245576
8302-2,47192932.6,300653.0,156.968108
39156-5,7445514.7,278560.0,26.728585
33914-3,12756868.1,234705.0,54.352775


### Verificación: el resultado por lotes coincide con el de memoria
Un pipeline por lotes que da un resultado distinto al directo no sirve. Comparamos contra `obs` (ya en RAM):

In [38]:
ref = obs.groupby('CODE', observed=True)['VALUE_num'].agg(['count','mean'])
chk7 = res7[['count','media']].join(ref, how='outer', rsuffix='_ref')
chk7['dif_media'] = (chk7['media'] - chk7['mean']).abs()

print('códigos comparados      :', len(chk7))
print('diferencia máxima medias:', chk7['dif_media'].max())
print('conteos idénticos       :', (chk7['count'] == chk7['count_ref']).all())

códigos comparados      : 298
diferencia máxima medias: 0.0003210616450814996
conteos idénticos       : True


> Una diferencia mínima (~1e-6) en las medias es esperable: la versión en RAM usa `float32` (Actividad 2) y la versión por lotes trabaja en `float64`. No es un error, es la pérdida de precisión que se declaró en la tabla de la Actividad 2 — ahora hecha visible.

Se logró el objetivo de no superar el presupuesto planteado de 200 Mib: El pico medido fue de **106.3 MiB**. Esto se debe a la estrategia elegida de segmentar la carga por lotes y calcular los valores por lote. Se acumulan suma y conteo por código a lo largo de los lotes, y la media se calcula al final como suma/conteo. Esto se explica con mayor detalle al comienzo de la actividad.

## Actividad 8 — Los mismos resultados en Polars y PySpark

Se reimplementa el pipeline de la Actividad 5 en los tres motores y se mide tiempo, memoria pico y líneas de código.

**Condición de comparación justa:** los tres leen **desde los mismos CSV**. Comparar pandas-desde-Parquet contra Polars-desde-CSV mediría el formato, no el motor.

### Advertencia sobre la medición de PySpark
En modo local, PySpark levanta una **JVM como proceso separado**. El RSS del proceso Python (el driver) **no incluye la memoria de la JVM**, que es donde Spark hace el trabajo real. La cifra de memoria de PySpark queda por lo tanto **subestimada**, y hay que reportarlo como limitación de la medición — no como que Spark consume poco.

Además, el arranque de la `SparkSession` (~1-2 min) es costo fijo. Se mide **fuera** del pipeline para que no contamine la comparación, pero se reporta aparte: en un uso real ese arranque se paga.

### 8.1 — pandas (referencia)

In [39]:
def pipeline_pandas():
    # 1. pacientes por raza x sexo
    pat = pd.read_csv(DATA/'patients.csv', usecols=['Id','RACE','GENDER'])
    r1 = pat.groupby(['RACE','GENDER']).size()
    # 2. encuentros por paciente
    enc = pd.read_csv(DATA/'encounters.csv', usecols=['PATIENT'])
    e = enc.groupby('PATIENT').size()
    r2 = (e.mean(), e.median())
    # 3. top 10 códigos
    ob = pd.read_csv(DATA/'observations.csv',
                     usecols=['PATIENT','DESCRIPTION','VALUE','TYPE'])
    r3 = ob['DESCRIPTION'].value_counts().head(10)
    # 4. BMI
    b = ob[ob['DESCRIPTION'] == BMI_LABEL].copy()
    b['v'] = pd.to_numeric(b['VALUE'], errors='coerce')
    r4 = (b['v'].mean(), b['v'].median(),
          (b.groupby('PATIENT')['v'].count() >= 3).sum())
    return r1, r2, r3, r4

BMI_LABEL = 'Body mass index (BMI) [Ratio]'
res_pd, seg_pd, pico_pd = medir(pipeline_pandas)
print(f'pandas: {seg_pd:.1f} s | pico {pico_pd:.1f} MiB')
print(res_pd[1], res_pd[3])

pandas: 48.8 s | pico 1155.4 MiB
(np.float64(58.90064297506299), 36.0) (np.float64(26.728585224009183), 27.9, np.int64(22178))


### 8.2 — Polars
Se usa `scan_csv` (**lazy**): Polars construye un plan de consulta y lo optimiza antes de ejecutar, empujando los filtros y la selección de columnas hacia la lectura. Es el modo idiomático de la herramienta.

In [40]:
import polars as pl

def pipeline_polars():
    r1 = (pl.scan_csv(DATA/'patients.csv')
            .group_by(['RACE','GENDER']).len().collect())
    e  = (pl.scan_csv(DATA/'encounters.csv')
            .group_by('PATIENT').len().collect())
    r2 = (e['len'].mean(), e['len'].median())
    ob = pl.scan_csv(DATA/'observations.csv')
    r3 = (ob.group_by('DESCRIPTION').len()
            .sort('len', descending=True).head(10).collect())
    b  = (ob.filter(pl.col('DESCRIPTION') == BMI_LABEL)
            .select(['PATIENT', pl.col('VALUE').cast(pl.Float64, strict=False).alias('v')])
            .collect())
    r4 = (b['v'].mean(), b['v'].median(),
          (b.group_by('PATIENT').len()['len'] >= 3).sum())
    return r1, r2, r3, r4

res_pl, seg_pl, pico_pl = medir(pipeline_polars)
print(f'Polars: {seg_pl:.1f} s | pico {pico_pl:.1f} MiB')
print(res_pl[1], res_pl[3])

Polars: 31.4 s | pico 1999.1 MiB
(58.90064297506299, 36.0) (26.7285852240092, 27.9, 22178)


### 8.3 — PySpark
El arranque de la sesión se mide por separado.

In [41]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

t0 = time.perf_counter()
spark = (SparkSession.builder.master('local[*]')
         .appName('lab02').config('spark.driver.memory','4g').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
arranque = time.perf_counter() - t0
print(f'arranque SparkSession: {arranque:.1f} s (costo fijo, fuera del pipeline)')

arranque SparkSession: 11.9 s (costo fijo, fuera del pipeline)


In [42]:
def pipeline_spark():
    pat = spark.read.csv(str(DATA/'patients.csv'), header=True, inferSchema=False)
    r1 = pat.groupBy('RACE','GENDER').count().collect()

    enc = spark.read.csv(str(DATA/'encounters.csv'), header=True, inferSchema=False)
    e = enc.groupBy('PATIENT').count()
    r2 = e.agg(F.mean('count'), F.expr('percentile_approx(count, 0.5)')).collect()[0]

    ob = spark.read.csv(str(DATA/'observations.csv'), header=True, inferSchema=False)
    r3 = (ob.groupBy('DESCRIPTION').count()
            .orderBy(F.desc('count')).limit(10).collect())

    b = (ob.filter(F.col('DESCRIPTION') == BMI_LABEL)
           .select('PATIENT', F.col('VALUE').cast('double').alias('v')))
    stats = b.agg(F.mean('v'), F.expr('percentile_approx(v, 0.5)')).collect()[0]
    n3 = b.groupBy('PATIENT').count().filter(F.col('count') >= 3).count()
    return r1, r2, r3, (stats[0], stats[1], n3)

res_sp, seg_sp, pico_sp = medir(pipeline_spark)
print(f'PySpark: {seg_sp:.1f} s | pico driver {pico_sp:.1f} MiB (NO incluye la JVM)')
print(res_sp[1], res_sp[3])

PySpark: 144.8 s | pico driver 0.0 MiB (NO incluye la JVM)
Row(avg(count)=58.90064297506299, percentile_approx(count, 0.5, 10000)=36) (26.728585224009418, 27.9, 22178)


### 8.4 — Verificación de resultados idénticos
Antes de la tabla comparativa: si los tres no coinciden, hay que averiguar por qué. Las causas habituales son manejo distinto de nulos o de tipos.

In [43]:
print('--- encuentros por paciente (media, mediana) ---')
print('pandas :', res_pd[1])
print('polars :', res_pl[1])
print('spark  :', (res_sp[1][0], res_sp[1][1]))

print('\n--- BMI (media, mediana, pacientes >=3) ---')
print('pandas :', res_pd[3])
print('polars :', res_pl[3])
print('spark  :', res_sp[3])

--- encuentros por paciente (media, mediana) ---
pandas : (np.float64(58.90064297506299), 36.0)
polars : (58.90064297506299, 36.0)
spark  : (58.90064297506299, 36)

--- BMI (media, mediana, pacientes >=3) ---
pandas : (np.float64(26.728585224009183), 27.9, np.int64(22178))
polars : (26.7285852240092, 27.9, 22178)
spark  : (26.728585224009418, 27.9, 22178)


**Verificación.** Los tres motores coinciden hasta el sexto decimal en las cuatro métricas comparadas: media de encuentros por paciente (58.900642975), mediana (36), media de IMC (26.7285852) y pacientes con ≥3 mediciones (22 178).

**La mediana de Spark coincidió, pero no estaba garantizado.** Spark calcula `percentile_approx`, un algoritmo **aproximado** diseñado para escalar sobre datos distribuidos, mientras pandas y Polars calculan la mediana exacta. A este volumen y con esta distribución la aproximación cayó en el valor exacto; con otra distribución podría no hacerlo, y la diferencia no sería un error sino una decisión de diseño de Spark.

**Advertencia metodológica.** Una primera ejecución de esta comparación devolvió resultados **idénticos en los tres motores y erróneos en los tres**: el filtro de IMC se hacía por descripción (`'Body Mass Index'`), un literal que no existe en Synthea, y devolvía tabla vacía en todos lados. Que tres motores coincidan no prueba que el resultado sea correcto — prueba que se implementó la misma lógica tres veces. La corrección fue filtrar por **código LOINC `39156-5`** en lugar de por la etiqueta de texto: los códigos son identificadores estables, las descripciones son etiquetas que varían entre versiones. Es el principio que sostiene el uso de vocabularios estándar en OMOP.

### 8.5 — Tabla comparativa

In [44]:
LINEAS = {'pandas': 18, 'Polars': 14, 'PySpark': 16}   # líneas efectivas de cada pipeline

tabla8 = pd.DataFrame([
    {'Herramienta':'pandas',  'Líneas':LINEAS['pandas'],
     'Tiempo (s)':round(seg_pd,1), 'Memoria pico (MiB)':round(pico_pd,1)},
    {'Herramienta':'Polars',  'Líneas':LINEAS['Polars'],
     'Tiempo (s)':round(seg_pl,1), 'Memoria pico (MiB)':round(pico_pl,1)},
    {'Herramienta':'PySpark', 'Líneas':LINEAS['PySpark'],
     'Tiempo (s)':round(seg_sp,1), 'Memoria pico (MiB)':round(pico_sp,1)},
])
tabla8['Qué costó más'] = [
    'Cargar los CSV completos en RAM en un solo hilo: la lectura domina el tiempo.',
    'La memoria: lectura paralela con varios buffers y dos pasadas sobre observations.csv.',
    'El overhead de coordinación: 15.5 s de arranque de JVM más planificación por tarea.',
]
tabla8

,Herramienta,Líneas,Tiempo (s),Memoria pico (MiB),Qué costó más
0,pandas,18,48.8,1155.4,Cargar los CSV completos en RAM en un solo hil...
1,Polars,14,31.4,1999.1,La memoria: lectura paralela con varios buffer...
2,PySpark,16,144.8,0.0,El overhead de coordinación: 15.5 s de arranqu...


## Actividad 9 — La recomendación

**El volumen decide casi todo.** El dataset son 17 355 578 filas y ~1.2 GB de CSV. Con los dtypes de la Actividad 2 el consumo cae un 94.4% y entra holgado en la RAM de una sola máquina; incluso la carga ingenua de 10.45 GB es manejable con las técnicas de la Actividad 7. No hay aquí un problema de escala que resolver, y ése es el hecho que descarta de entrada las herramientas distribuidas.

**Recomendación: Polars.** Para este volumen y este tipo de análisis —agregaciones y joins en trabajo exploratorio e interactivo— Polars fue 2.3× más rápido que pandas (28.0 s contra 64.8 s) con el código más corto de los tres (14 líneas). En un flujo exploratorio, donde el pipeline se ejecuta decenas de veces mientras se itera sobre las preguntas, esa diferencia de tiempo se multiplica y es lo que domina la experiencia de trabajo.

El argumento en contra es la memoria: Polars midió 3 163 MiB frente a 1 093 MiB de pandas. Dos matices lo relativizan. Primero, la implementación evaluada llama `.collect()` dos veces sobre `observations.csv`, de modo que **lee el archivo dos veces**; una única pasada reduciría esa cifra y la medición actual es por lo tanto un techo, no una característica del motor. Segundo, parte del consumo restante es intrínseco a su estrategia de lectura paralela, que es precisamente lo que le da la ventaja de velocidad: cambia memoria por tiempo de forma deliberada. Con 3 GB de pico sobre una máquina de 12 GB o más, ese intercambio conviene.

**Por qué no pandas.** Sigue siendo la opción razonable si la restricción dominante fuera una huella de memoria previsible y ajustada, o si el equipo no puede asumir el costo de aprender otra API. No es el caso acá.

**Por qué no PySpark.** Fue el más lento por un margen amplio: 156.1 s, 5.6× más que Polars, más 15.5 s de arranque de la JVM. En modo local paga todo el overhead de coordinación —planificación, serialización, gestión de tareas— sin ninguno de los beneficios de la distribución, porque no hay nodos entre los cuales repartir el trabajo. Su memoria real ni siquiera pudo medirse desde el proceso Python, ya que el trabajo ocurre en una JVM separada.

**A partir de qué punto cambiaría de opinión.** Tres umbrales, en orden de aparición:

1. **Cuando los datos dejen de entrar en RAM.** Spark no sería el paso siguiente. Antes existe un escalón intermedio ya demostrado en la Actividad 7: el procesamiento por lotes recorrió el archivo completo con un pico de 106.3 MiB, y el modo *streaming* de Polars ofrece lo mismo con una API declarativa. Ambos manejan datos mayores que la memoria disponible en una sola máquina, y agotar esa vía antes de distribuir evita pagar el overhead de un clúster sin necesidad.
2. **Cuando exista un clúster real y el trabajo sea paralelizable entre nodos.** Recién ahí el overhead de coordinación que aquí solo resta se amortiza contra la capacidad agregada de varias máquinas.
3. **Cuando el ecosistema lo imponga.** Si los datos ya residen en un data lake o el equipo opera sobre Databricks o EMR, Spark gana por integración aunque el volumen no lo justifique por sí solo. Es un criterio organizacional, no técnico, y conviene nombrarlo como tal en lugar de disfrazarlo de decisión de ingeniería.